In [ ]:
!git clone -b dev https://github.com/harishkulkarni10/ai-research-assistant-rag.git

# Dense embeddings for RAG
- We convert text chunks into dense vector embeddings

In [ ]:
# Load the chunks
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.parquet"

chunks_df = pd.read_parquet(CHUNKS_PATH)

print(f"Loaded {len(chunks_df):,} chunks")
chunks_df.head(2)

In [ ]:
# Installing embedding dependencies

!pip install -U sentence-transformers accelerate torch --quiet

# BGE embeddings
- Embed each chunk's text and store the resulting vectors alongside existing metadata


In [ ]:
# Load the BGE model

import torch
from sentence_transformers import SentenceTransformer, util

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', device)

bge_model = SentenceTransformer(
    'BAAI/bge-base-en-v1.5',
    device=device
)

In [ ]:
# Embedding chunks

texts = chunks_df['text'].tolist()

bge_embeddings = bge_model.encode(
    texts,
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings=True
)

print(f"Shape of the embeddings: {bge_embeddings.shape}")

In [ ]:
chunks_bge = chunks_df.copy()
chunks_bge['embedding'] = bge_embeddings.tolist()

output_path = PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet"
chunks_bge.to_parquet(output_path, index=False)

print(f"BGE embeddings saved to {output_path}")
chunks_bge.head(2)

# E5

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/ai-research-assistant-rag")
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.parquet"

chunks_df = pd.read_parquet(CHUNKS_PATH)

print(f"Loaded {len(chunks_df):,} chunks")
chunks_df.head(2)

In [ ]:
# Load E5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('using device: ', device)

e5_model = SentenceTransformer(
    'intfloat/e5-base-v2',
    device=device
)

In [ ]:
e5_texts = ['passage: ' + t for t in chunks_df['text'].tolist()]

print("Simple E5 input: ")
print(e5_texts[0][:200])

In [ ]:
# Generate E5 embeddings

e5_embeddings = e5_model.encode(
    e5_texts,
    batch_size = 32,
    show_progress_bar = True,
    normalize_embeddings=True
)

print(f"E5 embeddings shape: {e5_embeddings.shape}")

In [ ]:
chunks_e5 = chunks_df.copy()
chunks_e5["embedding"] = list(e5_embeddings)

output_path = PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet"
chunks_e5.to_parquet(output_path, index=False)

print(f"E5 embeddings saved to {output_path}")

In [ ]:
# Reload and verify
verify_df = pd.read_parquet(output_path)

print(f"Reloaded {len(verify_df):,} chunks")
print("Embedding dimension:", len(verify_df.iloc[0]["embedding"]))

# Let's compare these 2 embeddings as a sample for now - and then decide which embeddings to move forward with in the retrieval phase

In [ ]:
import numpy as np
from sentence_transformers.util import cos_sim

chunks_bge = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "chunks_bge.parquet")
bge_emb = np.array(chunks_bge['embedding'].tolist(), dtype=np.float32)

chunks_e5 = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "chunks_e5.parquet")
e5_emb = np.array(chunks_e5['embedding'].tolist(), dtype=np.float32)

# Sample queries
sample_queries = [
    "What are the key innovations in Transformer architecture?",
    "Explain self-attention mechanism in detail",
    "Recent advances in diffusion models",
    "How does RAG reduce hallucinations?",
    "Comparison between BERT and GPT models"
]

print("Sample Retrieval Comparison (Top-1 chunk preview)")

for query in sample_queries:
    print(f"\nQuery: {query}")

    # Embed query with both models
    q_bge = bge_model.encode(query, normalize_embeddings=True)
    q_e5 = e5_model.encode("query: " + query, normalize_embeddings=True)  # E5 uses prefix

    # Find top-1 for BGE
    sim_bge = cos_sim(q_bge, bge_emb)[0]
    top_idx_bge = np.argmax(sim_bge).item() # Extract integer value
    print(f"BGE Top-1 score: {sim_bge[top_idx_bge]:.4f}")
    print("BGE Top chunk preview:", chunks_bge.iloc[top_idx_bge]['text'][:300] + "...")

    # Find top-1 for E5
    sim_e5 = cos_sim(q_e5, e5_emb)[0]
    top_idx_e5 = np.argmax(sim_e5).item() # Extract integer value
    print(f"E5 Top-1 score: {sim_e5[top_idx_e5]:.4f}")
    print("E5 Top chunk preview:", chunks_e5.iloc[top_idx_e5]['text'][:300] + "...")